## STEP 1 - Install Java & Download Hadoop

In [ ]:
# Install Java 8 (Hadoop requires Java)
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!java -version

# Download Hadoop 3.3.6
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!mv hadoop-3.3.6 /usr/local/hadoop
print("Hadoop installed successfully")

## STEP 2 - Set Environment Variables

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['HADOOP_HOME'] = '/usr/local/hadoop'
os.environ['PATH'] = os.environ['PATH'] + ':/usr/local/hadoop/bin:/usr/local/hadoop/sbin'
os.environ['HADOOP_CONF_DIR'] = '/usr/local/hadoop/etc/hadoop'

# Verify
!hadoop version

## STEP 3 - Configure Hadoop for Local Standalone Mode

In [ ]:
# In local standalone mode, NO XML config changes are needed.
# Hadoop runs as a single Java process using local filesystem (not HDFS).
print("Local standalone mode requires no extra configuration.")
print("Hadoop will use local filesystem directly.")

## STEP 4 - Create a Sample System Log File

In [ ]:
!mkdir -p /content/log_input

log_data = """2024-01-01 10:00:01 INFO  UserService - User admin logged in successfully
2024-01-01 10:00:02 ERROR DBService - Database connection failed: timeout
2024-01-01 10:00:03 WARN  AuthService - Authentication token is expiring soon
2024-01-01 10:00:04 INFO  FileService - File upload completed successfully
2024-01-01 10:00:05 DEBUG Scheduler - Cron job triggered at 10:00:05
2024-01-01 10:00:06 ERROR NetworkService - Unable to reach host 192.168.1.1
2024-01-01 10:00:07 INFO  UserService - User john logged out
2024-01-01 10:00:08 WARN  DiskService - Disk usage above 80 percent
2024-01-01 10:00:09 INFO  APIService - GET /api/users returned 200 OK
2024-01-01 10:00:10 ERROR DBService - Query execution failed: syntax error
2024-01-01 10:00:11 DEBUG CacheService - Cache miss for key user_123
2024-01-01 10:00:12 INFO  AuthService - New session created for user alice
2024-01-01 10:00:13 WARN  MemoryService - Memory usage above 75 percent
2024-01-01 10:00:14 INFO  FileService - File download started by user bob
2024-01-01 10:00:15 ERROR AuthService - Login failed for user unknown_user
2024-01-01 10:00:16 DEBUG Scheduler - Background task completed in 120ms
2024-01-01 10:00:17 INFO  APIService - POST /api/login returned 200 OK
2024-01-01 10:00:18 WARN  NetworkService - High latency detected: 500ms
2024-01-01 10:00:19 INFO  DBService - Database backup completed successfully
2024-01-01 10:00:20 ERROR FileService - File not found: report_2024.pdf
2024-01-01 10:00:21 INFO  UserService - Password changed for user charlie
2024-01-01 10:00:22 DEBUG CacheService - Cache hit for key product_456
2024-01-01 10:00:23 WARN  DiskService - Disk cleanup recommended
2024-01-01 10:00:24 INFO  APIService - DELETE /api/session returned 200 OK
2024-01-01 10:00:25 ERROR NetworkService - SSL certificate expired
"""

with open('/content/log_input/system_log.txt', 'w') as f:
    f.write(log_data)

print("Log file created: system_log.txt")
print("\nFirst 5 lines of the log file:")
!head -5 /content/log_input/system_log.txt

## STEP 5 - Write LogMapper.java

The Mapper reads each log line, extracts the log level (INFO/ERROR/WARN/DEBUG), and emits `(log_level, 1)`.

In [ ]:
mapper_code = '''package com.logprocess;

import java.io.IOException;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.Mapper;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reporter;

public class LogMapper extends MapReduceBase
        implements Mapper<LongWritable, Text, Text, IntWritable> {

    private final static IntWritable one = new IntWritable(1);
    private Text logLevel = new Text();

    public void map(LongWritable key, Text value,
            OutputCollector<Text, IntWritable> output,
            Reporter reporter) throws IOException {
        String line = value.toString().trim();
        if (line.isEmpty()) return;
        // Log format: DATE TIME LEVEL SERVICE - MESSAGE
        // Index:       0    1    2     3       4  5...
        String[] parts = line.split("\\\\s+");
        if (parts.length >= 3) {
            logLevel.set(parts[2]);          // Extract log level at index 2
            output.collect(logLevel, one);   // Emit (log_level, 1)
        }
    }
}
'''

!mkdir -p /content/logprocess
with open('/content/logprocess/LogMapper.java', 'w') as f:
    f.write(mapper_code)
print("LogMapper.java written.")

## STEP 6 - Write LogReducer.java

The Reducer receives sorted `(log_level, 1)` pairs, sums up the counts for each log level, and emits `(log_level, total_count)`.

In [ ]:
reducer_code = '''package com.logprocess;

import java.io.IOException;
import java.util.Iterator;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reducer;
import org.apache.hadoop.mapred.Reporter;

public class LogReducer extends MapReduceBase
        implements Reducer<Text, IntWritable, Text, IntWritable> {

    public void reduce(Text key, Iterator<IntWritable> values,
            OutputCollector<Text, IntWritable> output,
            Reporter reporter) throws IOException {
        int sum = 0;
        while (values.hasNext()) {
            sum += values.next().get();   // Sum all 1s for this log level
        }
        output.collect(key, new IntWritable(sum));  // Emit (log_level, count)
    }
}
'''

with open('/content/logprocess/LogReducer.java', 'w') as f:
    f.write(reducer_code)
print("LogReducer.java written.")

## STEP 7 - Write LogRunner.java

In [ ]:
runner_code = '''package com.logprocess;

import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.FileInputFormat;
import org.apache.hadoop.mapred.FileOutputFormat;
import org.apache.hadoop.mapred.JobClient;
import org.apache.hadoop.mapred.JobConf;
import org.apache.hadoop.mapred.TextInputFormat;
import org.apache.hadoop.mapred.TextOutputFormat;

public class LogRunner {
    public static void main(String[] args) throws Exception {
        JobConf conf = new JobConf(LogRunner.class);
        conf.setJobName("LogProcessing");

        conf.setOutputKeyClass(Text.class);
        conf.setOutputValueClass(IntWritable.class);

        conf.setMapperClass(LogMapper.class);
        conf.setReducerClass(LogReducer.class);

        conf.setInputFormat(TextInputFormat.class);
        conf.setOutputFormat(TextOutputFormat.class);

        FileInputFormat.setInputPaths(conf, new Path(args[0]));
        FileOutputFormat.setOutputPath(conf, new Path(args[1]));

        JobClient.runJob(conf);
    }
}
'''

with open('/content/logprocess/LogRunner.java', 'w') as f:
    f.write(runner_code)
print("LogRunner.java written.")

## STEP 8 - Compile All Java Files

In [ ]:
!javac -classpath $(hadoop classpath) \
       -source 8 -target 8 \
       -d /content/logprocess \
       /content/logprocess/LogMapper.java \
       /content/logprocess/LogReducer.java \
       /content/logprocess/LogRunner.java

print("Compilation done.")

## STEP 9 - Create JAR File

In [ ]:
import os
os.chdir('/content/logprocess')
!jar -cvf logprocess.jar -C /content/logprocess .
print("JAR created.")

## STEP 10 - Run the MapReduce Job

In [ ]:
# Remove output dir if it exists from a previous run
!rm -rf /content/log_output

!hadoop jar /content/logprocess/logprocess.jar \
            com.logprocess.LogRunner \
            /content/log_input \
            /content/log_output 2>&1

print("MapReduce job completed.")

## STEP 11 - View the Output

In [ ]:
print("===== LOG PROCESSING OUTPUT (Log Level Counts) =====")
!cat /content/log_output/part-*

## STEP 12 - Visualize the Results

In [ ]:
import matplotlib.pyplot as plt

# Read output from local file
log_counts = {}
import glob
for fname in glob.glob('/content/log_output/part-*'):
    with open(fname) as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('\t')
                if len(parts) == 2:
                    log_counts[parts[0]] = int(parts[1])

print("Log Level Counts:")
for level, count in sorted(log_counts.items()):
    print(f"  {level:10s} : {count}")

# Bar chart
colors = {'INFO': 'steelblue', 'ERROR': 'crimson', 'WARN': 'orange', 'DEBUG': 'green'}
bar_colors = [colors.get(k, 'gray') for k in log_counts.keys()]

plt.figure(figsize=(8, 5))
plt.bar(log_counts.keys(), log_counts.values(), color=bar_colors, edgecolor='black')
plt.title('System Log Level Counts (MapReduce Output)', fontsize=14, fontweight='bold')
plt.xlabel('Log Level', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(fontsize=11)
for i, (k, v) in enumerate(log_counts.items()):
    plt.text(i, v + 0.1, str(v), ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('log_level_counts.png', dpi=150)
plt.show()
print("Chart saved as log_level_counts.png")

---
# IMPORTANT NOTES FOR ORAL EXAM
## DISTRIBUTED APPLICATION USING MapReduce - LOG FILE PROCESSING

---

## WHAT IS THIS PRACTICAL ABOUT?
This practical counts the number of occurrences of each **log level** (INFO, ERROR, WARN, DEBUG)  
from a system log file using **Hadoop MapReduce** — a distributed processing framework.

Three Java files used:
- `LogMapper.java`  — The Mapper (extracts log level, emits `(log_level, 1)`)
- `LogReducer.java` — The Reducer (sums counts per log level)
- `LogRunner.java`  — The Driver/Runner (configures and submits the job)

---

## WHAT IS MapReduce?
MapReduce is a programming model for processing large datasets in parallel across a cluster.

| Phase | What happens |
|---|---|
| **Map** | Each line is processed independently. Emits `(key, value)` pairs. |
| **Shuffle & Sort** | Framework groups all values for the same key together. |
| **Reduce** | Aggregates values for each key. Emits final `(key, result)`. |

---

## FULL MapReduce FLOW FOR THIS PRACTICAL

**INPUT (system_log.txt):**
```
2024-01-01 10:00:01 INFO  UserService - User logged in
2024-01-01 10:00:02 ERROR DBService - Connection failed
2024-01-01 10:00:03 WARN  AuthService - Token expiring
2024-01-01 10:00:04 INFO  FileService - Upload complete
```

**AFTER MAP PHASE (intermediate output):**
```
(INFO,  1)
(ERROR, 1)
(WARN,  1)
(INFO,  1)
```

**AFTER SHUFFLE & SORT (grouped by key):**
```
(DEBUG, [1, 1, 1])
(ERROR, [1, 1, 1, 1, 1])
(INFO,  [1, 1, 1, 1, 1, 1, 1, 1, 1])
(WARN,  [1, 1, 1, 1])
```

**AFTER REDUCE PHASE (final output):**
```
DEBUG    3
ERROR    5
INFO     9
WARN     4
```

---

## CODE EXPLANATION - LogMapper
```java
implements Mapper<LongWritable, Text, Text, IntWritable>
```
- Input Key   : LongWritable - byte offset of the line
- Input Value : Text         - one full log line
- Output Key  : Text         - log level (INFO/ERROR/WARN/DEBUG)
- Output Value: IntWritable  - always 1

```java
String[] parts = line.split("\\s+");  // Split by whitespace
logLevel.set(parts[2]);               // Index 2 = log level
output.collect(logLevel, one);        // Emit (log_level, 1)
```
- Log format: `DATE TIME LEVEL SERVICE - MESSAGE`
- `parts[0]` = date, `parts[1]` = time, `parts[2]` = log level

---

## CODE EXPLANATION - LogReducer
```java
implements Reducer<Text, IntWritable, Text, IntWritable>
```
- Receives `(log_level, [1, 1, 1, ...])` for each log level
- Sums all the 1s
- Emits `(log_level, total_count)`

```java
int sum = 0;
while (values.hasNext()) { sum += values.next().get(); }
output.collect(key, new IntWritable(sum));
```

---

## CODE EXPLANATION - LogRunner
```java
JobConf conf = new JobConf(LogRunner.class);   // Create job config
conf.setJobName("LogProcessing");              // Set job name
conf.setMapperClass(LogMapper.class);          // Set Mapper
conf.setReducerClass(LogReducer.class);        // Set Reducer
conf.setOutputKeyClass(Text.class);            // Output key type
conf.setOutputValueClass(IntWritable.class);   // Output value type
FileInputFormat.setInputPaths(conf, new Path(args[0]));   // Input path
FileOutputFormat.setOutputPath(conf, new Path(args[1]));  // Output path
JobClient.runJob(conf);                        // Submit and run job
```

---

## KEY COMMANDS USED
```bash
javac -classpath $(hadoop classpath) -source 8 -target 8 *.java  # Compile
jar -cvf logprocess.jar -C /content/logprocess .                 # Create JAR
hadoop jar logprocess.jar com.logprocess.LogRunner input output   # Run job
cat /content/log_output/part-*                                    # View output
```

---

## POSSIBLE ORAL EXAM QUESTIONS AND ANSWERS

**Q: What is this practical about?**  
A: It counts the number of occurrences of each log level (INFO, ERROR, WARN, DEBUG) from a system log file using Hadoop MapReduce distributed processing.

**Q: What does the Mapper do?**  
A: It reads each log line, splits it by whitespace, extracts the log level from index 2 (parts[2]), and emits (log_level, 1) as a key-value pair.

**Q: What does the Reducer do?**  
A: It receives all 1s grouped by log level (after shuffle & sort), sums them up, and emits (log_level, total_count) as the final output.

**Q: What is the Shuffle and Sort phase?**  
A: After the Map phase, Hadoop automatically groups all values with the same key together and sorts them alphabetically. The Reducer then processes each group.

**Q: Why do we use parts[2] to extract the log level?**  
A: The log format is DATE TIME LEVEL SERVICE - MESSAGE. After splitting by whitespace, index 0=date, 1=time, 2=log level. So parts[2] gives us INFO/ERROR/WARN/DEBUG.

**Q: Why do we use IntWritable instead of int?**  
A: Hadoop needs to serialize data to send it across the network. IntWritable implements the Writable interface for serialization. Regular Java int cannot be directly serialized by Hadoop.

**Q: What is a JAR file and why do we create one?**  
A: JAR = Java Archive. Packages all compiled .class files together. Hadoop needs a JAR to distribute and run the job across nodes.

**Q: Why do we use -source 8 -target 8 in javac?**  
A: Colab has Java 17 but Hadoop runs on Java 8. Without this flag, compiled bytecode is version 61 (Java 17) which Hadoop cannot load. These flags force compilation to Java 8 compatible bytecode.

**Q: What is a distributed application?**  
A: A distributed application runs across multiple computers (nodes) simultaneously, sharing the workload. Hadoop MapReduce distributes data processing across many nodes in a cluster for faster execution.

**Q: What is the expected output?**  
A: Each log level followed by its count, sorted alphabetically:
```
DEBUG    3
ERROR    5
INFO     9
WARN     4
```
